# Notebook 1: Setup and Indexing

Builds the ObliQA and RegulationQA indices from scratch.

**Outputs:**
- `faiss_index_obliqua.bin` — ObliQA FAISS index
- `clean_passages_obliqua.pkl` — ObliQA cleaned passages
- `bm25_obliqua.pkl` — ObliQA BM25 index
- `faiss_index_rqa.bin` — RegulationQA FAISS index
- `chunks_rqa.pkl` — RegulationQA chunks
- `bm25_rqa.pkl` — RegulationQA BM25 index

**Runtime:** ~5 min on T4 GPU

In [ ]:
# Install dependencies
!pip install -q sentence-transformers faiss-cpu rank-bm25 nltk pymupdf python-docx

In [ ]:
import requests, json, time, pickle, string, os
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import faiss, nltk

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

os.makedirs('/kaggle/working/index', exist_ok=True)
print('Dependencies ready.')

## Part 1: ObliQA Index

In [ ]:
# Load ObliQA documents and test data
print('Loading ObliQA...')
test_data = json.loads(requests.get(
    'https://raw.githubusercontent.com/RegNLP/ObliQADataset/main/ObliQA_test.json'
).text)
train_data = json.loads(requests.get(
    'https://raw.githubusercontent.com/RegNLP/ObliQADataset/main/ObliQA_train.json'
).text)
dev_data = json.loads(requests.get(
    'https://raw.githubusercontent.com/RegNLP/ObliQADataset/main/ObliQA_dev.json'
).text)
print(f'Train: {len(train_data)} Dev: {len(dev_data)} Test: {len(test_data)}')

all_passages = []
for doc_id in range(1, 41):
    r = requests.get(
        f'https://raw.githubusercontent.com/RegNLP/ObliQADataset/main/StructuredRegulatoryDocuments/{doc_id}.json'
    )
    if r.status_code == 200:
        all_passages.extend(json.loads(r.text))
    time.sleep(0.2)
print(f'Raw passages: {len(all_passages)}')

In [ ]:
# Clean ObliQA passages
clean_passages = []
for p in all_passages:
    text = p['Passage'].strip()
    if len(text) < 30:
        continue
    if len(text) > 1500:
        start = 0
        while start < len(text):
            chunk = text[start:start+1000]
            if len(chunk) < 100 and start > 0:
                break
            clean_passages.append({
                'DocumentID': p['DocumentID'],
                'PassageID':  p['PassageID'],
                'Passage':    chunk,
            })
            start += 800
    else:
        clean_passages.append({
            'DocumentID': p['DocumentID'],
            'PassageID':  p['PassageID'],
            'Passage':    text,
        })
print(f'Clean passages: {len(clean_passages)}')

In [ ]:
# Build ObliQA BGE index
print('Loading BGE...')
bge_model = SentenceTransformer('BAAI/bge-base-en-v1.5')

embeddings = bge_model.encode(
    [p['Passage'] for p in clean_passages],
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True
)
obliqua_index = faiss.IndexFlatIP(embeddings.shape[1])
obliqua_index.add(embeddings.astype(np.float32))
print(f'ObliQA FAISS index: {obliqua_index.ntotal} vectors')

# BM25
def tokenize(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text.split()

corpus = [tokenize(p['Passage']) for p in clean_passages]
obliqua_bm25 = BM25Okapi(corpus)
print('ObliQA BM25 ready.')

# Save
faiss.write_index(obliqua_index, '/kaggle/working/index/faiss_index_obliqua.bin')
with open('/kaggle/working/index/clean_passages_obliqua.pkl', 'wb') as f:
    pickle.dump(clean_passages, f)
with open('/kaggle/working/index/bm25_obliqua.pkl', 'wb') as f:
    pickle.dump(obliqua_bm25, f)
with open('/kaggle/working/index/test_data_obliqua.pkl', 'wb') as f:
    pickle.dump(test_data, f)
print('ObliQA index saved.')

## Part 2: RegulationQA Index

In [ ]:
import re
import pymupdf
import docx as python_docx
import pandas as pd

# Load RegulationQA dataset
rqa_df = pd.read_csv('/kaggle/input/regulationqa-artifacts/regulationqa_artifacts/regulationqa_v1.csv')
print(f'RegulationQA questions: {len(rqa_df)}')

# Document paths
doc_path = '/kaggle/input/datasets/sauraviitjapdcl/duomind-sshrag-mahalanobis/'

doc_name_map = {
    '24062022_Guidelines_on_Cyber_Security_in_Power_Sector_2021.pdf': 'Cyber Security Power Sector 2021',
    'CELEX_32016R0679_EN_TXT.pdf':               'GDPR',
    'CERT-In_Directions.docx':                   'CERT-In Directions 2022',
    'Cyber Security Framework in Banks.docx':    'RBI Cyber Security Framework',
    'DPDP ACT.pdf':                              'DPDP Act 2023',
    'DPDP_Rules_2025_English_only.pdf':          'Digital Personal Data Protection Rules 2025',
    'Electricity Act 2003.pdf':                  'Electricity Act 2003',
    'ISO_IEC-270012022-ed.3.pdf':                'ISO 27001:2022',
    'IT ACT 2000.pdf':                           'IT Act 2000',
    'NCIIPC-Rules-notification-1.pdf':           'IT Protected System Rules 2018',
    'NCIIPC_Guidelines_V2.pdf':                  'NCIIPC Guidelines',
    'NIST.SP.800-66r2.pdf':                      'NIST SP 800-66r2',
    'OECD-LEGAL-0479-en.docx':                   'OECD Digital Security Guidelines',
    'Presentation - CSCRF.pdf':                  'SEBI CSCRF',
    'State Data Centre security guidelines.pdf': 'State Data Centre security guidelines',
    'e-Governance Policy Initiatives.pdf':       'e-Governance Policy India',
    'guidelinesgovtentities.pdf':                'NCIIPC Guidelines for Govt Entities',
    'nist cyber security framework.docx':        'NIST Cybersecurity Framework',
    'pcissc_overview.pdf':                       'PCI Security Standards',
    'trai_takshalia.pdf':                        'Privacy, Security and Ownership of Data in the Telecom Sector Takshashila Policy Advisory',
}

def clean_text(text):
    text = re.sub(r'-\n', '', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()

def extract_pdf(filepath):
    doc = pymupdf.open(filepath)
    text = ''.join(page.get_text() for page in doc)
    doc.close()
    return clean_text(text)

def extract_docx(filepath):
    doc = python_docx.Document(filepath)
    return clean_text('\n'.join(p.text for p in doc.paragraphs))

print('Extracting documents...')
documents = {}
for fname, clean_name in doc_name_map.items():
    fpath = os.path.join(doc_path, fname)
    if not os.path.exists(fpath):
        print(f'  NOT FOUND: {fname}')
        continue
    try:
        text = extract_pdf(fpath) if fname.endswith('.pdf') else extract_docx(fpath)
        documents[clean_name] = text
        print(f'  OK: {clean_name} ({len(text.split()):,} words)')
    except Exception as e:
        print(f'  ERROR: {fname}: {e}')

print(f'Documents loaded: {len(documents)}')

In [ ]:
# Chunk documents
all_chunks = []
for doc_name, text in documents.items():
    words = text.split()
    start, idx = 0, 0
    while start < len(words):
        end = min(start + 500, len(words))
        chunk_text = ' '.join(words[start:end])
        if len(chunk_text) < 100 and start > 0:
            break
        if len(chunk_text) >= 30:
            all_chunks.append({
                'doc_name':  doc_name,
                'chunk_idx': idx,
                'text':      chunk_text,
            })
            idx += 1
        start += 400  # 100-word overlap

print(f'Total chunks: {len(all_chunks)}')
from collections import Counter
for doc, count in Counter(c['doc_name'] for c in all_chunks).most_common():
    print(f'  {doc}: {count} chunks')

In [ ]:
# Build RegulationQA index
rqa_embeddings = bge_model.encode(
    [c['text'] for c in all_chunks],
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)
rqa_index = faiss.IndexFlatIP(rqa_embeddings.shape[1])
rqa_index.add(rqa_embeddings.astype(np.float32))
print(f'RegulationQA FAISS index: {rqa_index.ntotal} vectors')

rqa_corpus = [tokenize(c['text']) for c in all_chunks]
rqa_bm25   = BM25Okapi(rqa_corpus)
print('RegulationQA BM25 ready.')

faiss.write_index(rqa_index, '/kaggle/working/index/faiss_index_rqa.bin')
with open('/kaggle/working/index/chunks_rqa.pkl', 'wb') as f:
    pickle.dump(all_chunks, f)
with open('/kaggle/working/index/bm25_rqa.pkl', 'wb') as f:
    pickle.dump(rqa_bm25, f)
print('RegulationQA index saved.')

In [ ]:
# Summary
print('\n=== INDEX SUMMARY ===')
print(f'ObliQA passages:       {obliqua_index.ntotal:,}')
print(f'RegulationQA chunks:   {rqa_index.ntotal:,}')
print(f'ObliQA test questions: {len(test_data):,}')
print(f'RegulationQA questions:{len(rqa_df):,}')
print('\nAll indices saved to /kaggle/working/index/')